# M3 · ① SIF 교차분석과 TF-IDF 기준선

`03_M3_SIF_기초_EDA.ipynb`에서 확인한 기인물·작업 분포를 더 살피는 후속 분석입니다. 실제 입력은 `source_01_sif_cases.parquet`입니다. TF-IDF는 **한국어 띄어쓰기 기반 탐색용 기준선**이며, 단어 빈도가 원인이나 위험률을 뜻하지 않습니다.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), None)
assert ROOT is not None, 'VS Code에서 SANUP-P 폴더를 열고 커널을 .venv로 선택하세요.'
PERSONAL = ROOT / 'data' / 'personal'
RAW = PERSONAL / 'raw' / 'source_01' / 'source_01_sif_archive.xlsx'
PROCESSED = PERSONAL / 'processed' / 'source_01_sif_cases.parquet'
assert RAW.exists(), f'① SIF 원본이 없습니다: {RAW}'
PROCESSED.parent.mkdir(parents=True, exist_ok=True)
print('원본:', RAW.name, f'{RAW.stat().st_size:,} bytes')

assert PROCESSED.exists(), 'M2 SIF 전처리를 먼저 실행하세요.'
cases = pd.read_parquet(PROCESSED)
manufacturing = cases[cases['sheet_group'] == '제조업 등']
construction = cases[cases['sheet_group'] == '건설업']
print('제조업 등:', len(manufacturing), '건설업:', len(construction))

## 1. 범주 교차표

표가 지나치게 커지지 않도록 각각 상위 10개 범주만 표시합니다. 나머지를 0건으로 해석하지 않습니다.

In [ ]:
m_rows = manufacturing['hazard_work'].value_counts().head(10).index
m_cols = manufacturing['causal_object'].value_counts().head(10).index
m_subset = manufacturing[manufacturing['hazard_work'].isin(m_rows) & manufacturing['causal_object'].isin(m_cols)]
display(pd.crosstab(m_subset['hazard_work'], m_subset['causal_object']))

c_rows = construction['work_name'].value_counts().head(10).index
c_cols = construction['accident_type'].value_counts().head(10).index
c_subset = construction[construction['work_name'].isin(c_rows) & construction['accident_type'].isin(c_cols)]
display(pd.crosstab(c_subset['work_name'], c_subset['accident_type']))

## 2. 재해유발요인 TF-IDF

사례의 `trigger_factor` 문장을 한 문서로 사용합니다. 한국어 형태소 분석을 하지 않은 간단한 기준선이므로 조사·동형어 문제가 남습니다.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

texts = cases['trigger_factor'].dropna().astype(str)
vectorizer = TfidfVectorizer(
    token_pattern=r'(?u)\b[가-힣A-Za-z0-9]{2,}\b',
    ngram_range=(1, 2), min_df=3, max_features=300,
)
matrix = vectorizer.fit_transform(texts)
scores = pd.Series(matrix.mean(axis=0).A1, index=vectorizer.get_feature_names_out(), name='평균 TF-IDF')
top_terms = scores.sort_values(ascending=False).head(30)
display(top_terms.to_frame())
print('문서 수:', matrix.shape[0], '특징 수:', matrix.shape[1])

## 3. 개인 결과 저장·해석

상위 단어가 원본 문장과 어떤 관계인지 사례 몇 건을 직접 확인하고, 인과나 발생률처럼 해석하지 않습니다.

In [ ]:
target = PERSONAL / 'reports' / 'source_01_tfidf_terms.csv'
target.parent.mkdir(parents=True, exist_ok=True)
top_terms.to_csv(target, encoding='utf-8-sig', header=True)
print('저장:', target)

## TF-IDF 상위 표현 그래프

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import Image, display

ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None
if any(font.name=='Malgun Gothic' for font in font_manager.fontManager.ttflist):
    plt.rcParams['font.family']='Malgun Gothic'
plt.rcParams['axes.unicode_minus']=False
terms=pd.read_csv(ROOT/'data'/'personal'/'reports'/'source_01_tfidf_terms.csv',encoding='utf-8-sig')
selected=terms.iloc[:15].iloc[::-1]
fig,ax=plt.subplots(figsize=(9,6))
ax.barh(selected.iloc[:,0].astype(str),selected.iloc[:,1].astype(float),color='#4f7183')
ax.set(xlabel='평균 TF-IDF',title='① SIF · 재해유발요인 상위 표현')
fig.tight_layout()
path=ROOT/'data'/'personal'/'figures'/'sif_tfidf_top15.png'
path.parent.mkdir(parents=True,exist_ok=True)
fig.savefig(path,dpi=170,bbox_inches='tight');plt.close(fig)
display(Image(filename=str(path)))

## 텍스트 분석 결과

- 분석 대상은 SIF 6,032건의 `재해유발요인` 열이다. `재해개요`나 사고 원인 라벨을 모델에 넣은 결과가 아니다.
- 공백 기준 한국어 토큰을 대상으로 unigram·bigram TF-IDF를 계산했다. 상위 표현에 `추락`, `작업`, `위험`, `위해`, `잃고`, `깔림`이 나타났다. `작업`, `위해`처럼 일반적인 표현도 상위라 용어 정리 없이 원인 순위로 읽기 어렵다.
- 형태소 분석기, 불용어 사전, 도메인 용어 통일은 적용하지 않았다. 따라서 동의어·조사·띄어쓰기 차이가 남아 있는 탐색용 기준선이다. TF-IDF 점수는 사례 수나 위험도, 원인 기여율을 뜻하지 않는다.
- 결과 표는 `data/personal/reports/source_01_tfidf_terms.csv`, 그래프는 `data/personal/figures/` 아래 개인 산출물이다. 특정 상위 단어를 안전 대책으로 바로 연결하지 않는다.

### 사용할 때 주의할 점

- 보고서나 발표에서 원인·예방대책을 말할 때는 검색어 빈도만 인용하지 말고 사례 ID와 원문 필드를 확인한다.
- ①은 출처표시·변경금지 조건이 붙은 자료이므로 공개 재배포나 변형물 공유 전에 이용 조건을 다시 확인한다.
- 정교한 키워드 분석은 ⑦ 용어사전 기반 정규화 전후를 별도 실험으로 비교하고, 같은 문서·같은 평가 기준을 유지한다.
